In [3]:
# evaluate_field_plots_corrected_v6.py
#
# Purpose: Evaluate 390-tile final models on 50 field plot tiles
# Summer model evaluated on summer tiles
# Fall model evaluated on fall tiles — CORRECTED from previous run
# which incorrectly used summer tiles for fall model evaluation

from ultralytics import YOLO
import pandas as pd
import yaml
import os
import shutil

# ── paths ──────────────────────────────────────────────────────────────────────
SUMMER_WEIGHTS = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\final_summer_yolo_v6_390\final\weights\best.pt"
FALL_WEIGHTS   = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\final_fall_yolo_v6_390\final\weights\best.pt"

SUMMER_IMAGES  = r"F:\Masterthesis\labeling\field_plots\summer"
FALL_IMAGES    = r"F:\Masterthesis\labeling\field_plots\fall"
LABELS_DIR     = r"F:\Masterthesis\labeling\field_plots\summer\fieldplot_labels_v5_remapped"

OUTPUT_DIR = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── fix label filenames — remove UUID prefix ───────────────────────────────────
LABELS_FIXED = os.path.join(OUTPUT_DIR, "labels")
os.makedirs(LABELS_FIXED, exist_ok=True)

print("Fixing label filenames...")
for fname in os.listdir(LABELS_DIR):
    if not fname.endswith(".txt"):
        continue
    clean = fname.split("-", 1)[1] if "-" in fname else fname
    shutil.copy2(
        os.path.join(LABELS_DIR, fname),
        os.path.join(LABELS_FIXED, clean)
    )
print(f"Fixed {len(os.listdir(LABELS_FIXED))} label files")

# ── evaluation function ────────────────────────────────────────────────────────
def evaluate(weights_path, images_dir, season):
    print(f"\n{'='*60}")
    print(f"Evaluating {season.upper()} model on {season} field plot tiles")
    print(f"{'='*60}")

    # set up season-specific image and label folders
    eval_dir    = os.path.join(OUTPUT_DIR, season)
    eval_images = os.path.join(eval_dir, "images")
    eval_labels = os.path.join(eval_dir, "labels")
    os.makedirs(eval_images, exist_ok=True)
    os.makedirs(eval_labels, exist_ok=True)

    # copy correct season images
    for fname in os.listdir(images_dir):
        if fname.endswith(".png"):
            shutil.copy2(
                os.path.join(images_dir, fname),
                os.path.join(eval_images, fname)
            )

    # copy fixed labels
    for fname in os.listdir(LABELS_FIXED):
        shutil.copy2(
            os.path.join(LABELS_FIXED, fname),
            os.path.join(eval_labels, fname)
        )

    # create yaml
    yaml_path = os.path.join(eval_dir, "dataset.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump({
            "path":  eval_dir,
            "train": eval_images,
            "val":   eval_images,
            "nc":    2,
            "names": ["deciduous_tree", "shrub"]
        }, f, default_flow_style=False)

    model = YOLO(weights_path)
    metrics = model.val(
        data     = yaml_path,
        imgsz    = 640,
        batch    = 4,
        device   = "cpu",
        project  = OUTPUT_DIR,
        name     = f"eval_{season}",
        exist_ok = True,
        save_txt = True,
        save_conf= True,
        plots    = True,
        verbose  = True,
    )

    precision = metrics.box.mp
    recall    = metrics.box.mr
    map50     = metrics.box.map50
    map5095   = metrics.box.map
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    ap50      = metrics.box.ap50
    ap50_dec  = float(ap50[0]) if len(ap50) > 0 else None
    ap50_shr  = float(ap50[1]) if len(ap50) > 1 else None

    print(f"\n=== {season.upper()} FIELD PLOT RESULTS ===")
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}")
    print(f"mAP50:               {map50:.4f}")
    print(f"mAP50-95:            {map5095:.4f}")
    print(f"F1:                  {f1:.4f}")
    print(f"deciduous_tree AP50: {ap50_dec:.4f}")
    print(f"shrub AP50:          {ap50_shr:.4f}")

    return {
        "season":         season,
        "dataset":        "50 field plot tiles",
        "n_tiles":        50,
        "precision":      round(precision, 4),
        "recall":         round(recall, 4),
        "mAP50":          round(map50, 4),
        "mAP50_95":       round(map5095, 4),
        "F1":             round(f1, 4),
        "AP50_deciduous": round(ap50_dec, 4) if ap50_dec else None,
        "AP50_shrub":     round(ap50_shr, 4) if ap50_shr else None,
    }

# ── run evaluations ────────────────────────────────────────────────────────────
summer_res = evaluate(SUMMER_WEIGHTS, SUMMER_IMAGES, "summer")
fall_res   = evaluate(FALL_WEIGHTS,   FALL_IMAGES,   "fall")

# ── comparison ─────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("COMPARISON: CV → Test Set → Field Plots")
print(f"{'='*60}")
print(f"\nSUMMER:")
print(f"  CV mAP50:          0.5904")
print(f"  Field plot mAP50:  {summer_res['mAP50']:.4f}")
print(f"\nFALL:")
print(f"  CV mAP50:          0.2369")
print(f"  Field plot mAP50:  {fall_res['mAP50']:.4f}")

# ── save ───────────────────────────────────────────────────────────────────────
rows = [
    {"model": "YOLOv8m v6 390-tile", **summer_res},
    {"model": "YOLOv8m v6 390-tile", **fall_res},
]
csv_path = os.path.join(OUTPUT_DIR, "field_plot_results_v2.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

Fixing label filenames...
Fixed 50 label files

Evaluating SUMMER model on summer field plot tiles
Ultralytics 8.4.107  Python-3.11.14 torch-2.10.0+cpu CPU (AMD Ryzen 7 5700U with Radeon Graphics)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.50.1 ms, read: 131.420.5 MB/s, size: 730.1 KB)
val: Scanning C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6\summer\labels... 50 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 115.4it/s 0.4s.2s
val: New cache created: C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6\summer\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.8s/it 36.3s2.7s
                   all         50        577      0.554      0.517      0.506      0.214
        deciduous_tree         25        340      0.539       0.52      0.526      0.224